In [40]:
!pip install pymupdf

   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   -- ------------------------------------- 1.0/19.2 MB 10.9 MB/s eta 0:00:02
   ------ --------------------------------- 3.1/19.2 MB 11.1 MB/s eta 0:00:02
   ----------- ---------------------------- 5.5/19.2 MB 11.0 MB/s eta 0:00:02
   ---------------- ----------------------- 7.9/19.2 MB 11.0 MB/s eta 0:00:02
   --------------------- ------------------ 10.5/19.2 MB 11.1 MB/s eta 0:00:01
   ------------------------- -------------- 12.3/19.2 MB 10.7 MB/s eta 0:00:01
   ----------------------------- ---------- 14.4/19.2 MB 10.6 MB/s eta 0:00:01
   ---------------------------------- ----- 16.8/19.2 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------  19.1/19.2 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------- 19.2/19.2 MB 10.5 MB/s  0:00:01



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import faiss
from langchain_classic.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_classic.vectorstores import FAISS

c:\Users\thaku\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\thaku\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
loader = PyMuPDFLoader(file_path = r"D:\live_Projects\langchain\data\ml_ebook.pdf" )

In [36]:
docs = loader.load()

In [37]:
len(docs)

279

In [38]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 100
)

chunks = splitter.split_documents(docs)
len(chunks)

466

In [7]:
model_name = 'sentence-transformers/all-MiniLM-L6-V2'
embedding_model = HuggingFaceEmbeddings(model_name = model_name)

In [40]:
texts = []
for chunk in chunks:
    texts.append(chunk.page_content)

In [41]:
len(texts)

466

In [21]:
index = faiss.IndexFlatL2(384)

In [ ]:
from langchain_community.docstore.in_memory import InMemoryDocstore

vector_store = FAISS(
    embedding_function = embedding_model,
    index = index,
    docstore = InMemoryDocstore(),
    index_to_docstore_id = {}
)

In [42]:
vector_store = FAISS.from_texts(texts = texts, embedding = embedding_model)

In [43]:
vector_store.save_local('faiss_db')

In [8]:
vector_store = FAISS.load_local(r"D:\live_Projects\langchain\Rag\vector_stores\faiss_db", embeddings=embedding_model, allow_dangerous_deserialization=True)

In [45]:
vector_store

In [46]:
query = input("Explain machine learning ? ")
vector_store.similarity_search(query=query, k = 10)

[Document(id='851a6be5-9623-4a45-9df9-b2f4cd665eb8', metadata={}, page_content='ing algorithm. The system is shown mostly normal instances during training, so it\nlearns to recognize them and when it sees a new instance it can tell whether it looks\n18 \n| \nChapter 1: The Machine Learning Landscape'),
 Document(id='2c3cee23-fd06-4952-8abc-ec174ad2a4ba', metadata={}, page_content='Figure 1-11. Semisupervised learning\nMost semisupervised learning algorithms are combinations of unsupervised and\nsupervised algorithms. For example, deep belief networks (DBNs) are based on unsu‐\npervised components called restricted Boltzmann machines (RBMs) stacked on top of\none another. RBMs are trained sequentially in an unsupervised manner, and then the\nwhole system is fine-tuned using supervised learning techniques.\nReinforcement Learning\nReinforcement Learning is a very different beast. The learning system, called an agent\nin this context, can observe the environment, select and perform action

In [47]:
vector_store.similarity_search_with_score(query=query, k = 10)

[(Document(id='851a6be5-9623-4a45-9df9-b2f4cd665eb8', metadata={}, page_content='ing algorithm. The system is shown mostly normal instances during training, so it\nlearns to recognize them and when it sees a new instance it can tell whether it looks\n18 \n| \nChapter 1: The Machine Learning Landscape'),
  np.float32(1.2753415)),
 (Document(id='2c3cee23-fd06-4952-8abc-ec174ad2a4ba', metadata={}, page_content='Figure 1-11. Semisupervised learning\nMost semisupervised learning algorithms are combinations of unsupervised and\nsupervised algorithms. For example, deep belief networks (DBNs) are based on unsu‐\npervised components called restricted Boltzmann machines (RBMs) stacked on top of\none another. RBMs are trained sequentially in an unsupervised manner, and then the\nwhole system is fine-tuned using supervised learning techniques.\nReinforcement Learning\nReinforcement Learning is a very different beast. The learning system, called an agent\nin this context, can observe the environmen

In [9]:
# retriever 
retreiver = vector_store.as_retriever(
    search_type = 'mmr',
    search_kwargs = {
        'k' : 100,
        'fetch_k' : 200,
        "lambda_mult" : 0.8
    }
)

In [49]:
retreived_content = retreiver.invoke(query)

In [10]:
from langchain_classic.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

In [11]:
template = PromptTemplate(
    template = '''
You are an expert pdf ai assistant. Answer the following questions only from the given context
do not hallucinate if you didn't get the enough relevence or context
Question : {question} 
Context : {context}, 
Answer : ''', 
input_variables = ['question', 'context']
)

In [12]:
def content(data):
    context = ''
    for doc in data:
        context += doc.page_content
    return context

In [53]:
content(retreived_content)

'ing algorithm. The system is shown mostly normal instances during training, so it\nlearns to recognize them and when it sees a new instance it can tell whether it looks\n18 \n| \nChapter 1: The Machine Learning LandscapeFigure 1-11. Semisupervised learning\nMost semisupervised learning algorithms are combinations of unsupervised and\nsupervised algorithms. For example, deep belief networks (DBNs) are based on unsu‐\npervised components called restricted Boltzmann machines (RBMs) stacked on top of\none another. RBMs are trained sequentially in an unsupervised manner, and then the\nwhole system is fine-tuned using supervised learning techniques.\nReinforcement Learning\nReinforcement Learning is a very different beast. The learning system, called an agent\nin this context, can observe the environment, select and perform actions, and get\nrewards in return (or penalties in the form of negative rewards, as in Figure 1-12). It\nmust then learn by itself what is the best strategy, called a 

In [ ]:
prompt = template.invoke({'question' : query, 'context' :content(retreived_content) })

In [55]:
print(prompt)

text="\nYou are an expert pdf ai assistant. Answer the following questions only from the given context\ndo not hallucinate if you didn't get the enough relevence or context\nQuestion : What is Generative ai \nContext : ing algorithm. The system is shown mostly normal instances during training, so it\nlearns to recognize them and when it sees a new instance it can tell whether it looks\n18 \n| \nChapter 1: The Machine Learning LandscapeFigure 1-11. Semisupervised learning\nMost semisupervised learning algorithms are combinations of unsupervised and\nsupervised algorithms. For example, deep belief networks (DBNs) are based on unsu‐\npervised components called restricted Boltzmann machines (RBMs) stacked on top of\none another. RBMs are trained sequentially in an unsupervised manner, and then the\nwhole system is fine-tuned using supervised learning techniques.\nReinforcement Learning\nReinforcement Learning is a very different beast. The learning system, called an agent\nin this context,

In [15]:
parallel_chain = RunnableParallel(
    {'question' :  RunnablePassthrough(),
    'context': retreiver  | RunnableLambda(content)}
)

In [65]:
parallel_chain.invoke(query)

{'question': 'What is Generative ai',
 'context': 'ing algorithm. The system is shown mostly normal instances during training, so it\nlearns to recognize them and when it sees a new instance it can tell whether it looks\n18 \n| \nChapter 1: The Machine Learning LandscapeFigure 1-11. Semisupervised learning\nMost semisupervised learning algorithms are combinations of unsupervised and\nsupervised algorithms. For example, deep belief networks (DBNs) are based on unsu‐\npervised components called restricted Boltzmann machines (RBMs) stacked on top of\none another. RBMs are trained sequentially in an unsupervised manner, and then the\nwhole system is fine-tuned using supervised learning techniques.\nReinforcement Learning\nReinforcement Learning is a very different beast. The learning system, called an agent\nin this context, can observe the environment, select and perform actions, and get\nrewards in return (or penalties in the form of negative rewards, as in Figure 1-12). It\nmust then le

In [16]:
from dotenv import load_dotenv
load_dotenv()
llm = HuggingFaceEndpoint(
    repo_id = 'meta-llama/Meta-Llama-3-8B-Instruct',
    task = 'text-generation',
    max_new_tokens = 100,
    temperature = 0.5
)

model = ChatHuggingFace(llm = llm)

In [19]:
from langchain_ollama import ChatOllama
model = ChatOllama(model = 'mistral')

In [20]:
chain = parallel_chain | template | model
result = chain.invoke(input("enter the query : "))
print(result.content)

 In the provided text, several machine learning concepts and algorithms are discussed, primarily focused on supervised learning. Here's a summary of some key points:

1. **Data Preparation**: This involves cleaning, transforming, and organizing data for use in machine learning models.

2. **Scikit-Learn**: A popular open-source Python library used for machine learning tasks. It provides various algorithms for regression, classification, clustering, etc.

3. **Linear Regression**: A simple yet powerful statistical model used to predict a continuous outcome based on one or more predictors (independent variables). Scikit-Learn offers both Ordinary Least Squares (OLS) and Ridge Regression for linear models.

4. **Logistic Regression**: Used for binary classification problems, where the dependent variable is categorical with only two possible values. It estimates probabilities of each class given a set of predictors.

5. **Support Vector Machines (SVM)**: A popular machine learning algorith

In [60]:
# chain.invoke(query)